# Поиск объявлений услуг Авито

Нужно найти до 50 объявлений для каждого запроса. Сначала попробуем поиск по тексту, потом добавим CatBoost и проверим, помогает ли их смесь. Качество сравниваем по Recall@50: какую долю выбранных пользователями объявлений удалось вернуть, в среднем по запросам.

### 1. Подготовка

Подключаем библиотеки, указываем пути к трём файлам и папку для результатов. SEED фиксируем, чтобы при повторном запуске выбирать те же запросы для проверки и обучения.

Для запуска нужны pandas, NumPy, scikit-learn, CatBoost, tqdm и движок для Parquet, например pyarrow. Используем открытые библиотеки, TF-IDF и смешивание списков по местам объявлений. Запросов к внешним API в решении нет. Ячейки запускаем сверху вниз; для обучения CatBoost в этом коде нужна NVIDIA GPU, а пути к данным при запуске у себя нужно заменить.

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from IPython.display import display

TRAIN_PATH = Path("/kaggle/input/datasets/shaykoaleksandr/dataset-files/train.parquet")
QUERIES_PATH = Path("/kaggle/input/datasets/shaykoaleksandr/dataset-files/benchmark_queries.parquet")
ITEMS_PATH = Path("/kaggle/input/datasets/shaykoaleksandr/dataset-files/benchmark_items.parquet")

# Тут сохранятся результаты
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 67
rng = np.random.default_rng(SEED)

### 2. Загрузка данных

Читаем файлы в три таблицы: train с пользовательскими выборами, queries с запросами для ответа и items с объявлениями, среди которых нужно искать. Заодно выводим размеры, чтобы проверить, что всё загрузилось.

In [ ]:
start = time.time()

train = pd.read_parquet(TRAIN_PATH)
queries = pd.read_parquet(QUERIES_PATH)
items = pd.read_parquet(ITEMS_PATH)

for name, df in [
    ("train", train),
    ("queries", queries),
    ("items", items),
]:
    print(
        f"{name}: {len(df):,} строк, "
        f"{df.shape[1]} колонок, "
    )

train: 497,673 строк, 19 колонок, 
queries: 2,452 строк, 6 колонок, 
items: 189,212 строк, 14 колонок, 


### 3. Пропуски и типы колонок

Загрузились 497 673 обучающие пары, 2 452 запроса и 189 212 объявлений. Теперь смотрим, где есть пропуски и в каком виде записаны признаки. Это пригодится, когда будем готовить текст и числовые поля.

In [ ]:
for name, df in [
    ("train", train),
    ("queries", queries),
    ("items", items),
]:
    print(f"\n{name}")

    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "missing_percent": (100 * df.isna().mean()).round(2),
    })

    display(summary)


train


,dtype,missing_count,missing_percent
search_query,object,0,0.00
search_location_id,int64,0,0.00
search_is_delivery_search,int32,0,0.00
search_infm_params_text,object,0,0.00
search_category,int64,0,0.00
item_title_raw,object,0,0.00
item_rating_reviews_count,float64,19211,3.86
item_rating,float64,28232,5.67
item_price,object,0,0.00
item_microcat_id,int64,0,0.00



queries


,dtype,missing_count,missing_percent
query_id,object,0,0.0
search_query,object,0,0.0
search_location_id,int64,0,0.0
search_is_delivery_search,int32,0,0.0
search_infm_params_text,object,0,0.0
search_category,int64,0,0.0



items


,dtype,missing_count,missing_percent
item_title_raw,object,0,0.00
item_rating_reviews_count,float64,11615,6.14
item_rating,float64,17631,9.32
item_price,object,0,0.00
item_microcat_id,int64,0,0.00
item_longitude,object,1,0.00
item_location_id,int64,0,0.00
item_latitude,object,1,0.00
item_is_phone_hidden,bool,0,0.00
item_is_message_forbidden,bool,0,0.00


### 4. Какие объявления встречаются в обеих выборках

В запросах пропусков нет. В объявлениях иногда не хватает рейтинга и числа отзывов, а у 35 объявлений benchmark нет описания. Теперь проверим, сколько объявлений из train вообще есть среди тех, которые разрешено возвращать в ответе.

In [ ]:
corpus_ids = set(items["item_id"])
train_in_corpus = train["item_id"].isin(corpus_ids)

print(
    "\nДоля строк train с объявлением из benchmark_items: "
    f"{train_in_corpus.mean():.2%}"
)

unique_train_ids = set(train["item_id"].dropna())
print(f"Уникальных объявлений в train: {len(unique_train_ids):,}")
print(
    "Из них есть в benchmark_items: "
    f"{len(unique_train_ids & corpus_ids):,}"
)


Доля строк train с объявлением из benchmark_items: 6.63%
Уникальных объявлений в train: 344,825
Из них есть в benchmark_items: 18,142


### 5. Повторяющиеся запросы

В benchmark нашлись 18 142 из 344 825 уникальных объявлений train. На них приходится 6,63% обучающих строк, поэтому одного запоминания выбранных объявлений не хватит.

Теперь смотрим на запросы. Один текст в разных городах или с разными фильтрами считаем разными комбинациями признаков поиска. Собираем такие комбинации в query_groups и отдельно проверяем совпадения с benchmark по всем признакам и только по тексту.

In [ ]:
search_cols = [
    column for column in queries.columns
    if column.startswith("search_")
]

print("Признаки запроса:", search_cols)
print(f"\nУникальных текстов в train: {train['search_query'].nunique():,}")
print(f"Уникальных текстов в benchmark: {queries['search_query'].nunique():,}")

# Одна группа — одинаковый набор всех search-признаков.
# dropna=False сохраняет группы с незаполненными фильтрами.

# Здесь мы формируем группы из запросов. Одна группа - все одинаковые search признаки. Ну и пишем dropna=False, чтобы сохранять
# группы у которых не заполнены фильтры.
query_groups = (
    train.groupby(search_cols, dropna=False, observed=True)
    .agg(
        rows=("item_id", "size"),
        relevant_items=("item_id", "nunique"),
    )
    .reset_index()
)

print(f"Уникальных полных запросов в train: {len(query_groups):,}")

print("\nСколько выбранных объявлений приходится на один полный запрос:")
display(
    query_groups["relevant_items"]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .to_frame()
)

# Здесь мы проверяем, что у нас полностью совпадают запросы по признакам из трейна и из бенчмарка
unique_train_queries = train[search_cols].drop_duplicates()

query_overlap = queries[search_cols].merge(
    unique_train_queries,
    on=search_cols,
    how="left",
    indicator=True,
    validate="many_to_one",
)

print(
    "\nДоля benchmark-запросов, полностью совпадающих с запросами train: "
    f"{query_overlap['_merge'].eq('both').mean():.2%}"
)

print(
    "Доля benchmark-запросов с текстом, встречавшимся в train: "
    f"{queries['search_query'].isin(train['search_query']).mean():.2%}"
)

Признаки запроса: ['search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category']

Уникальных текстов в train: 74,529
Уникальных текстов в benchmark: 2,452
Уникальных полных запросов в train: 354,463

Сколько выбранных объявлений приходится на один полный запрос:


,relevant_items
count,354463.000000
mean,1.317638
std,1.611180
min,1.000000
50%,1.000000
90%,2.000000
95%,3.000000
99%,6.000000
max,278.000000



Доля benchmark-запросов, полностью совпадающих с запросами train: 4.40%
Доля benchmark-запросов с текстом, встречавшимся в train: 36.99%


### 6. Примеры запросов и объявлений

Около 37% текстов benchmark уже встречались в train, но по всем признакам поиска совпадает только 4,4% запросов. Посмотрим несколько реальных примеров и частые категории, чтобы понять, что именно записано в текстах и параметрах.

In [ ]:
with pd.option_context("display.max_colwidth", 220):
    print("Примеры запросов benchmark:")
    display(queries.sample(min(5, len(queries)), random_state=SEED))

    print("\nПримеры положительных пар из train:")
    example_cols = search_cols + [
        column for column in [
            "item_id",
            "item_title_raw",
            "item_description_raw",
            "item_infm_params_text",
            "item_category_id",
            "item_location_id",
        ]
        if column in train.columns
    ]

    display(
        train[example_cols].sample(
            min(5, len(train)),
            random_state=SEED,
        )
    )

print("\nКатегории запросов в train:")
display(train["search_category"].value_counts(dropna=False).head(15))

print("\nКатегории объявлений в корпусе:")
display(items["item_category_id"].value_counts(dropna=False).head(15))

Примеры запросов benchmark:


,query_id,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category
1412,kNwxx7DHbGAJHsLi,вывоз старой мебели на свалку,639440,0,Вид услуги Вывоз мусора и вторсырья,114
316,2KzUhe61bPyE1wuH,утепление крыши минватой,635860,0,,114
1017,vVsNRyPU9Jus2tEX,бригада по ремонту домов,628780,0,,0
1353,N9mTnUXVm03Yoijm,перевозка мебели,656901,0,Вид услуги Грузоперевозки,114
476,YVhrad8cmxvFxAji,сахьянова 5,653271,0,"Тип услуги Ресницы, брови Вид услуги Красота, здоровье",114



Примеры положительных пар из train:


,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_id,item_title_raw,item_description_raw,item_infm_params_text,item_category_id,item_location_id
120327,компрессор воздушный аренда,662760,0,"Вид услуги Оборудование, производство",114,af5b912f8d36ac13,Аренда бензо- и электроинструмента,Аренда бензо- и электроинструмента почасовая/посуточная.,"Вид услуги Оборудование, производство Тип услуги Аренда оборудования Место оказания услуг Ярославская область, Рыбинск, Заволжский планировочный район, район Заволжье-1 Тип стоимости за услугу Начальная цена График р...",114,662760
64519,регулировка клапанов,635320,0,"Вид услуги Автосервис, аренда",114,1d5b4f2f4ea59055,"Ремонт мототехники, эндуро, снегоходы, квадроциклы","Cпециaлизиpуемся нa peмонте, сбоpке, тeхничeском oбслуживании и подборке эндуpо/кpoccoвых мотоциклoв, дoрожных мотоциклoв, квадроциклов и снегоходов.\r\nPаботaем c тexникoй из Китaя, Японии, Aвстрии.\n\nУcлуги:\r\n•...","Вид услуги Автосервис, аренда Тип услуги Автосервис Место оказания услуг Красноярск, улица Калинина Гарантия Опыт работы 11 Дополнительно Тёплый бокс Тип стоимости за услугу Начальная цена Время для связи от 39600 Вр...",114,635320
8487,автоэлектрик,637640,0,"Тип услуги автосервиса Диагностика и ремонт авто Вид услуги Автосервис, аренда Тип услуги Автосервис",114,f279d884761ff3f9,"Автоэлектрик GM (Opel, Chevrolet, Cadillac )","Диагностика и ремонт электрооборудования автомобилей марки GM (Opel, Chevrolet , Cadillac). Большой опыт работы , с 2012 по настоящее время работаю в должности диагноста в дилерском центре GM. Гарантия качества, отв...","Вид услуги Автосервис, аренда Тип услуги Автосервис Место оказания услуг Москва, Кавказский бульвар, 54с5 Гарантия Опыт работы 10 Дополнительно Тёплый бокс Ремонт двигателя Диагностика двигателя Ремонт двигателя Заме...",114,637640
464898,гостиничные чеки,637640,0,,114,5d2620ea37a52b7d,Отчетные документы для командировки и подбор квар,"Oтпpaвьте нам cообщение 📩 с подpобнoстями вaшeго запpoса нa кpaткocрочную аpeнду квaртиры, гoстиницы или жилья для комaндировoк в Mocквe 🏙.\n\nПри необхoдимости мы гoтовы предоcтавить слeдующие oтчeтныe комaндиpовочн...","Вид услуги Деловые услуги Тип услуги Бухгалтерия, финансы Место оказания услуг Москва, проезд Воскресенские Ворота Тип стоимости за услугу Начальная цена Опыт работы 10 лет и больше Берёте ли срочные заказы Как вы ра...",114,637640
213389,автоэлектрик,624360,0,"Тип услуги автосервиса Диагностика и ремонт авто Вид услуги Автосервис, аренда Тип услуги Автосервис",114,1766935d306ef247,Выездной автоэлектрик. Компьютерная диагностика,Выезжаю по городу и области. Без выходных.\n\nЧто сделаю:\n\n✔ Компьютерная диагностика любой системы\n\n✔ Поиск скрытых неисправностей\n\n✔ Устранение проблем на месте\n\n✔ Гарантия на работы\n\nЛюбые марки авто. \n...,"Вид услуги Автосервис, аренда Тип услуги Автосервис Место оказания услуг Тракторная ул., 11 Гарантия Опыт работы 1 Тип стоимости за услугу Начальная цена Время для связи от 27000 Время для связи до 77400 Время для св...",114,624360



Категории запросов в train:


search_category
114    497635
0          34
81          2
10          2
Name: count, dtype: int64


Категории объявлений в корпусе:


item_category_id
114    187336
19        357
112       141
40        122
33        103
10         98
20         84
25         82
36         74
111        74
39         72
24         70
83         68
27         57
30         56
Name: count, dtype: int64

### Что видно по данным

В train получилось 354 463 разных комбинации признаков поиска. Для большинства известно одно выбранное объявление, в среднем около 1,32. Примерно 37% текстов benchmark уже встречались в train, но по всем признакам совпадает только 4,4% запросов.

Почти все обучающие пары и объявления benchmark относятся к категории 114. Поэтому дальше опираемся на текст и проверяем локацию. Просто запомнить пары из train недостаточно: большая часть объявлений benchmark там не встречалась.

### 7. Проверка локации

Посчитаем, как часто пользователь выбирал объявление из той же локации, где искал. Отдельно посмотрим запросы с доставкой и без неё. Это поможет решить, стоит ли давать объявлениям из своей локации больший вес.

In [ ]:
same_location = (
    train["search_location_id"] == train["item_location_id"]
)

print(
    "Доля обучающих пар с одинаковой локацией запроса и объявления:",
    f"{same_location.mean():.2%}"
)

location_stats = (
    train.assign(same_location=same_location)
    .groupby("search_is_delivery_search")
    .agg(
        pairs=("item_id", "size"),
        same_location_share=("same_location", "mean"),
    )
)

display(location_stats)

print("\nСамые частые локации поиска:")
display(train["search_location_id"].value_counts().head(10))

Доля обучающих пар с одинаковой локацией запроса и объявления: 83.10%


,pairs,same_location_share
search_is_delivery_search,,
0,497661,0.831028
1,12,0.500000



Самые частые локации поиска:


search_location_id
637640    36276
653240    25229
107620    12277
633540    11441
650400    10720
654070     8204
641780     8145
625810     8106
652000     7963
646600     7706
Name: count, dtype: int64

### 8. Запросы для валидации

Локации совпали в 83,10% обучающих пар. Значит, этот признак стоит проверить в поиске. По доставке всего 12 строк, так что делать отдельные выводы по ней пока не будем.

Выбираем 1 000 разных текстов, для каждого оставляем одну случайную комбинацию локации и остальных признаков. В valid_truth собираем объявления, которые пользователи выбирали именно по этим запросам. Из train_fit исключаем все строки с выбранными текстами, даже если локация другая. Так проверяем поиск по текстам, которых не будет в обучении CatBoost.

In [ ]:
# Тут для одного текста мы оставляем только один какой-то запрос полный.
validation_pool = (
    query_groups
    .sample(frac=1, random_state=SEED)
    .drop_duplicates("search_query")
)

N_VALID = min(1000, len(validation_pool))

valid_queries = (
    validation_pool
    .sample(n=N_VALID, random_state=SEED)
    [search_cols]
    .reset_index(drop=True)
)

# Присвоим номера запросам, потому что в train их нет.
valid_queries.insert(0, "val_id", np.arange(len(valid_queries)))

# Тут для выбранных валидационных запросов находим объявления, которые пользователи выбирали по ним в train (совпадают все признаки)
valid_pairs = train.merge(
    valid_queries,
    on=search_cols,
    how="inner",
    validate="many_to_one",
)

valid_truth = (
    valid_pairs.groupby("val_id")["item_id"]
    .agg(lambda values: set(values))
    .reindex(valid_queries["val_id"])
    .tolist()
)

# Убираем из обучения все строки с нашими выбранными валидационными текстами.
heldout_texts = set(valid_queries["search_query"])

# Чтобы не занимать лишнюю память, пока что оставим выбранные колонки объявлений. Если что, тексты есть в train и сможем замерджить
fit_columns = search_cols + [
    "item_id",
    "item_microcat_id",
    "item_location_id",
]

train_fit = train.loc[
    ~train["search_query"].isin(heldout_texts),
    fit_columns,
].copy()

print(f"Валидационных запросов: {len(valid_queries):,}")
print(f"Положительных пар для них: {len(valid_pairs):,}")
print(f"Строк в обучающей части: {len(train_fit):,}")
print(
    "Среднее число релевантных объявлений:",
    round(np.mean([len(ids) for ids in valid_truth]), 2),
)

Валидационных запросов: 1,000
Положительных пар для них: 1,093
Строк в обучающей части: 490,305
Среднее число релевантных объявлений: 1.07


### 9. Среди каких объявлений проверяем поиск

Для валидации получилось 1 093 положительные пары, в среднем 1,07 разных выбранных объявлений на запрос. Некоторые из этих объявлений отсутствуют в benchmark_items. Если их не добавить, алгоритм просто не сможет найти правильный ответ.

Создаём retrieval_items: сначала все объявления benchmark, затем недостающие объявления валидации. Алгоритм увидит их тексты и признаки, но не получит подсказку, для какого запроса они правильные. При подготовке итогового CSV добавленные объявления исключаем. Такая проверка удобна для сравнения методов, но её результат не обязан совпадать с оценкой на скрытом тесте.

In [ ]:
# Здесь мы совместим бенчарковские объявления с нашими валидационными, первыми пойдут benchmark_items, а потом уже валидационные
items = items.reset_index(drop=True)

extra_valid_items = (
    valid_pairs.loc[
        ~valid_pairs["item_id"].isin(items["item_id"]),
        items.columns,
    ]
    .drop_duplicates("item_id")
)

# вот собственно они и есть
retrieval_items = pd.concat(
    [items, extra_valid_items],
    ignore_index=True,
)

retrieval_ids = retrieval_items["item_id"].to_numpy()
retrieval_locations = retrieval_items["item_location_id"].to_numpy()

corpus_id_set = set(retrieval_ids)

print(f"Объявлений в benchmark: {len(items):,}")
print(f"Добавлено для валидации: {len(extra_valid_items):,}")
print(f"Всего в локальном корпусе: {len(retrieval_items):,}")

Объявлений в benchmark: 189,212
Добавлено для валидации: 990
Всего в локальном корпусе: 190,202


### 10. Подготовка текста

Добавили 990 объявлений, всего для локальной проверки получилось 190 202. Теперь приводим тексты к нижнему регистру, заменяем ё на е и убираем лишние символы. В словах с кириллицей заменяем похожие латинские буквы; слова целиком на латинице оставляем.

В document_texts объединяем заголовок, параметры и первые 2 500 символов описания. Заголовок повторяем дважды, чтобы увеличить его вклад. Отдельно сохраняем title_texts для поиска по частям слов в заголовке.

In [ ]:
import re
import unicodedata

# На всякий случай заменим латинские буквы, которые похожи на русские внутри слов, где уже есть русские буквы. англ названия сохраняем
lookalikes = str.maketrans({
    "a": "а",
    "c": "с",
    "e": "е",
    "o": "о",
    "p": "р",
    "x": "х",
    "y": "у",
    "k": "к",
    "m": "м",
    "t": "т",
    "b": "в",
    "h": "н",
})

word_pattern = re.compile(r"[a-zа-я0-9]+")
cyrillic_pattern = re.compile(r"[а-я]")

def normalize_text(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value)).lower()
    text = text.replace("ё", "е")

    words = word_pattern.findall(text)

    return " ".join(
        word.translate(lookalikes)
        if cyrillic_pattern.search(word)
        else word
        for word in words
    )

start = time.time()

title_texts = (
    retrieval_items["item_title_raw"]
    .fillna("")
    .map(normalize_text)
)

# Повтор заголовка увеличивает его вес внутри текстового представления.
document_texts = (
    retrieval_items["item_title_raw"].fillna("") + " "
    + retrieval_items["item_title_raw"].fillna("") + " "
    + retrieval_items["item_infm_params_text"].fillna("") + " "
    + retrieval_items["item_description_raw"].fillna("").str.slice(0, 2500)
).map(normalize_text)

print(f"Подготовка текстов: {time.time() - start:.1f} сек.")
print("\nПример заголовка:", title_texts.iloc[0])

Подготовка текстов: 106.6 сек.

Пример заголовка: ремонт выкуп компьют и ноутбуков с выездом на дом


### 11. Переводим тексты в TF-IDF

Строим два числовых представления объявлений. X_words учитывает отдельные слова из общего текста, а X_chars учитывает кусочки длиной от 3 до 5 символов в заголовке. Второй вариант помогает с некоторыми словоформами и опечатками. Матрицы хранятся в разреженном виде, чтобы не занимать лишнюю память.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

start = time.time()

word_vectorizer = TfidfVectorizer(
    lowercase=False,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.98,
    max_features=180_000,
    sublinear_tf=True,
    dtype=np.float32,
)

X_words = word_vectorizer.fit_transform(document_texts)

print(
    f"Словарный индекс готов: {X_words.shape}, "
    f"{time.time() - start:.1f} сек."
)

char_vectorizer = TfidfVectorizer(
    lowercase=False,
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_features=120_000,
    sublinear_tf=True,
    dtype=np.float32,
)

X_chars = char_vectorizer.fit_transform(title_texts)

print(
    f"Символьный индекс готов: {X_chars.shape}, "
    f"общее время {time.time() - start:.1f} сек."
)

Словарный индекс готов: (190202, 180000), 40.0 сек.
Символьный индекс готов: (190202, 79259), общее время 48.6 сек.


### 12. Первый поиск и Recall@50

Для каждого запроса считаем сходство по словам и добавляем сходство по частям слов с весом 0,4. Затем пробуем множители 1, 2 и 4 для объявлений с совпадающей локацией. Объявления из других локаций не исключаем.

Берём 50 лучших и сравниваем с valid_truth. Метрика сначала считает долю найденных правильных объявлений для каждого запроса, а потом усредняет эти доли.

In [ ]:
from tqdm.auto import tqdm

def recall_at_50(predictions, truth):
    return float(np.mean([
        len(set(pred) & relevant) / len(relevant)
        for pred, relevant in zip(predictions, truth)
    ]))


def retrieve_tfidf(
    query_df,
    location_multipliers=(1.0, 2.0, 4.0),
    benchmark_only=False,
    batch_size=32,
):
    # Для получения baseline используем только сам запрос. Фильтры потом добавим отдельным экспериментом
    query_texts = query_df["search_query"].map(normalize_text)


    Q_words = word_vectorizer.transform(query_texts)
    Q_chars = char_vectorizer.transform(query_texts)

    # В финальном ответе ищем уже только среди benchmark_items
    n_docs = len(items) if benchmark_only else len(retrieval_items)

    W = X_words[:n_docs].T.tocsr()
    C = X_chars[:n_docs].T.tocsr()

    ids = retrieval_ids[:n_docs]
    locations = retrieval_locations[:n_docs]
    query_locations = query_df["search_location_id"].to_numpy()

    results = {m: [] for m in location_multipliers}
    k = min(50, n_docs)

    for start in tqdm(range(0, len(query_df), batch_size)):
        end = min(start + batch_size, len(query_df))

        # Тут считаем сходство запроса и объявления по словам
        scores = (Q_words[start:end] @ W).toarray()
        # Сходство по частям слов в заголовке добавляем с весом 0.4, чтобы найти разные формы слов и опечатки
        scores += 0.4 * (Q_chars[start:end] @ C).toarray()

        for row, base_scores in enumerate(scores):
            same_loc = (
                locations == query_locations[start + row]
            )

            for multiplier in location_multipliers:
                adjusted = base_scores.copy()
                adjusted[same_loc] *= multiplier

                # Тут выбираем 50 лучших без полной сортировки корпуса
                top = np.argpartition(adjusted, -k)[-k:]
                top = top[np.argsort(-adjusted[top], kind="stable")]

                results[multiplier].append(ids[top].tolist())

    return results


valid_predictions = retrieve_tfidf(valid_queries)

validation_results = pd.DataFrame([
    {
        "location_multiplier": multiplier,
        "Recall@50": recall_at_50(predictions, valid_truth),
    }
    for multiplier, predictions in valid_predictions.items()
]).sort_values("Recall@50", ascending=False)

display(validation_results)

  0%|          | 0/32 [00:00<?, ?it/s]

,location_multiplier,Recall@50
2,4.0,0.740100
1,2.0,0.650600
0,1.0,0.335667


### 13. Сохраняем baseline

Без учёта локации получили Recall@50 = 0,3357. Множитель 2 дал 0,6506, а множитель 4 дал 0,7401. Из проверенных вариантов выбираем 4.

Теперь запускаем поиск для запросов benchmark, уже только среди исходных объявлений items. Проверяем ID, повторы и число кандидатов, затем сохраняем answer_baseline.csv. Это первый готовый ответ, который оставим для сравнения.

In [ ]:
best_multiplier = float(
    validation_results.iloc[0]["location_multiplier"]
)

benchmark_predictions = retrieve_tfidf(
    queries,
    location_multipliers=(best_multiplier,),
    benchmark_only=True,
)[best_multiplier]

answer = pd.DataFrame({
    "query_id": queries["query_id"],
    "answer": [" ".join(ids) for ids in benchmark_predictions],
})

benchmark_id_set = set(items["item_id"])

assert list(answer.columns) == ["query_id", "answer"]
assert len(answer) == len(queries)
assert answer["query_id"].is_unique
assert set(answer["query_id"]) == set(queries["query_id"])

for prediction in benchmark_predictions:
    assert 0 < len(prediction) <= 50
    assert len(prediction) == len(set(prediction))
    assert all(item_id in benchmark_id_set for item_id in prediction)

answer.to_csv(
    OUTPUT_DIR / "answer_baseline.csv",
    index=False,
    encoding="utf-8",
)

print("Сохранён:", OUTPUT_DIR / "answer_baseline.csv")
print("Лучший множитель локации:", best_multiplier)
display(answer.head(3))

  0%|          | 0/77 [00:00<?, ?it/s]

Сохранён: /kaggle/working/answer_baseline.csv
Лучший множитель локации: 4.0


,query_id,answer
0,70DfDUpwjxB4lzFd,b92ee8f432cec2d1 deb58d7ad8c27d06 6042fecdd775...
1,JTrdTaZJvSiLPkXj,cbeccbecb1fb8d86 16ea26ff0aa0450a 367af128a9ea...
2,LZCZNoVG4AFUkVRJ,3f89b8062dc85f1c d8fce513e4f000a7 dab52187b450...


### 14. Готовим запросы для CatBoost

Baseline получил 0,690 на Stepik. Попробуем улучшить отбор 50 объявлений с помощью CatBoost.

Из train_fit берём пары, где выбранное объявление есть в retrieval_items: для него уже рассчитан TF-IDF. Собираем правильные ответы по каждому набору признаков поиска и выбираем до 3 000 запросов. Это быстрая обучающая выборка по доступному пересечению, а не весь train. Валидационные тексты в неё не входят.

In [ ]:
from catboost import CatBoostClassifier
import gc

# Берем из train_fit только те пары, объявления которых есть в retrieval. Для них уже расчитан TF-IDF
available_train = train_fit[
    train_fit["item_id"].isin(retrieval_ids)
]

# Тут собираем множество выбранных объявлений, которые известны в retrieval для каждого запроса (комбинация текста, локации, фильтров)
cb_groups = (
    available_train
    .groupby(search_cols, dropna=False, observed=True)["item_id"]
    .agg(lambda values: set(values))
    .reset_index(name="relevant_ids")
)

cb_groups = cb_groups.sample(
    n=min(3000, len(cb_groups)),
    random_state=SEED,
).reset_index(drop=True)

cb_queries = cb_groups[search_cols].copy()
cb_truth = cb_groups["relevant_ids"].tolist()

print("Запросов для обучения CatBoost:", len(cb_queries))

Запросов для обучения CatBoost: 3000


### 15. Числовые признаки объявлений

Подготовим рейтинг, цену, число отзывов и признаки доступности связи. Пропущенный рейтинг заменяем на -1, а пропуски в цене и числе отзывов на 0. К цене и числу отзывов применяем log(1 + x), чтобы уменьшить разброс больших значений.

FEATURE_NAMES хранит названия будущих признаков CatBoost. Кроме полей объявления, туда войдут текстовые сходства и совпадение локации и категории.

In [ ]:
def numeric_values(column, default):
    return (
        pd.to_numeric(retrieval_items[column], errors="coerce")
        .fillna(default)
        .to_numpy(dtype=np.float32)
    )

item_rating_values = numeric_values("item_rating", -1)
item_review_values = np.log1p(
    np.maximum(numeric_values("item_rating_reviews_count", 0), 0)
)
item_price_values = np.log1p(
    np.maximum(numeric_values("item_price", 0), 0)
)

item_categories = retrieval_items["item_category_id"].to_numpy()

item_phone_hidden = (
    retrieval_items["item_is_phone_hidden"]
    .fillna(False).to_numpy(dtype=np.float32)
)
item_messages_forbidden = (
    retrieval_items["item_is_message_forbidden"]
    .fillna(False).to_numpy(dtype=np.float32)
)

FEATURE_NAMES = [
    "word_similarity",
    "char_similarity",
    "filter_similarity",
    "same_location",
    "text_score",
    "text_score_location4",
    "rating",
    "log_reviews",
    "log_price",
    "same_category",
    "phone_hidden",
    "messages_forbidden",
]

### 16. Собираем больше кандидатов

Для каждого запроса берём по 150 лучших объявлений при множителях локации 1, 4 и 16. Объединяем списки и удаляем повторы, получается не больше 450 кандидатов. Для каждой пары запроса и кандидата собираем признаки для CatBoost.

Текст фильтров сравниваем с текстом объявления, но выполнение условий вроде «рейтинг от 4» этим не проверяем. Категорию запроса 0 считаем отсутствием ограничения; это наше предположение о значении поля.

В конце считаем, какую долю правильных ответов нашли среди всех кандидатов. Это верхняя граница для следующего этапа: CatBoost не сможет вернуть объявление, которое сюда не попало.

In [ ]:
def top_indices(scores, k):
    k = min(k, len(scores))
    selected = np.argpartition(scores, -k)[-k:]
    return selected[
        np.argsort(-scores[selected], kind="stable")
    ]

def build_candidate_packs(query_df, benchmark_only=False, batch_size=32):
    n_docs = len(items) if benchmark_only else len(retrieval_items)

    # Теперь берем TF-IDF нужных объявлений и для локальной проверки всего retrieval, а для отправки только из benchmark

    W = X_words[:n_docs].T.tocsr()
    C = X_chars[:n_docs].T.tocsr()

    query_texts = query_df["search_query"].map(normalize_text)
    filter_texts = query_df["search_infm_params_text"].map(normalize_text)

    Qw = word_vectorizer.transform(query_texts)
    Qc = char_vectorizer.transform(query_texts)
    Qf = word_vectorizer.transform(filter_texts)

    q_locations = query_df["search_location_id"].to_numpy()
    q_categories = query_df["search_category"].to_numpy()

    packs = []

    for start in tqdm(range(0, len(query_df), batch_size)):
        end = min(start + batch_size, len(query_df))

        word_scores = (Qw[start:end] @ W).toarray()
        char_scores = (Qc[start:end] @ C).toarray()

        # Тут сравниваем слова из фильтров поиска со словами в объявлении. Их сходство передадим в CatBoost отдельным признаком.
        # При этом мы не сравниваем числовые условия на сходство (например, «рейтинг от 4»)
        filter_scores = (Qf[start:end] @ W).toarray()

        for row in range(end - start):
            query_number = start + row

            base = word_scores[row] + 0.4 * char_scores[row]
            same_location = (
                retrieval_locations[:n_docs] == q_locations[query_number]
            )

            candidates = np.unique(np.concatenate([
                top_indices(base, 150),
                top_indices(base * (1 + 3 * same_location), 150),
                top_indices(base * (1 + 15 * same_location), 150),
            ]))

            same = same_location[candidates].astype(np.float32)
            text = base[candidates]

            # наш признак равен единице, если категории совпадают. Если search_category = 0, то считаем, что
            # ограничений нет и поэтому на совпадение тоже ставим признаку 1
            category_match = (
                (q_categories[query_number] == 0)
                | (item_categories[candidates] == q_categories[query_number])
            ).astype(np.float32)

            features = np.column_stack([
                word_scores[row, candidates],
                char_scores[row, candidates],
                filter_scores[row, candidates],
                same,
                text,
                text * (1 + 3 * same),
                item_rating_values[candidates],
                item_review_values[candidates],
                item_price_values[candidates],
                category_match,
                item_phone_hidden[candidates],
                item_messages_forbidden[candidates],
            ]).astype(np.float32)

            packs.append({
                "indices": candidates,
                "features": features,
            })

    return packs


print("Кандидаты для обучения:")
train_packs = build_candidate_packs(cb_queries)

print("Кандидаты для валидации:")
valid_packs = build_candidate_packs(valid_queries)

# теперь проверяем, какую долю правильных ответов нашёл первый этап. CatBoost будет выбирать только из этих кандидатов, потому
# пропущенные первым этапом объвления он уже не сможет просматривать

candidate_coverage = np.mean([
    len(set(retrieval_ids[pack["indices"]]) & truth) / len(truth)
    for pack, truth in zip(valid_packs, valid_truth)
])

print(f"Полнота расширенного списка кандидатов: {candidate_coverage:.4f}")

Кандидаты для обучения:


  0%|          | 0/94 [00:00<?, ?it/s]

Кандидаты для валидации:


  0%|          | 0/32 [00:00<?, ?it/s]

Полнота расширенного списка кандидатов: 0.8520


### 17. Обучаем CatBoost

Расширенные списки нашли 85,20% правильных ответов в среднем по запросам. Теперь учим модель выбирать из них 50. Ставим метку 1 известным выбранным объявлениям и 0 остальным кандидатам. Ноль означает, что выбор не наблюдался, а не что объявление точно не подходит.

При обучении пропускаем запросы, где первый этап не нашёл ни одного правильного ответа. На валидации такие запросы остаются. Каждому обучающему запросу даём одинаковый общий вес, независимо от числа кандидатов. Обучаем 500 деревьев на GPU, сохраняем модель и смотрим важность признаков.

In [ ]:
train_feature_blocks = []
train_label_blocks = []
train_weight_blocks = []

skipped_queries = 0

for pack, truth in zip(train_packs, cb_truth):
    candidate_ids = retrieval_ids[pack["indices"]]

    labels = np.array(
        [item_id in truth for item_id in candidate_ids],
        dtype=np.int32,
    )

    if labels.sum() == 0:
        skipped_queries += 1
        continue

    train_feature_blocks.append(pack["features"])
    train_label_blocks.append(labels)

    # Мы делим общий вес запроса поровну между его объявлениями. Это нужно, потому что запросы с большим количество кандидатов
    # не получают больший суммарный вес в обучении просто из-за числа строк
    train_weight_blocks.append(
        np.full(len(labels), 1.0 / len(labels), dtype=np.float32)
    )

X_cb_train = np.vstack(train_feature_blocks)
y_cb_train = np.concatenate(train_label_blocks)
weights_cb = np.concatenate(train_weight_blocks)

# Тут приводим средний вес строки к 1
weights_cb /= weights_cb.mean()

print("Обучающих пар:", len(y_cb_train))
print("Положительных пар:", y_cb_train.sum())
print("Пропущено запросов без найденных положительных:", skipped_queries)

cb_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.06,
    loss_function="Logloss",
    task_type="GPU",
    devices="0",
    random_seed=SEED,
    verbose=100,
    allow_writing_files=False,
)

# Правильные ответы для валидации в обучение не передаем
cb_model.fit(
    X_cb_train,
    y_cb_train,
    sample_weight=weights_cb,
)

cb_model.save_model(str(OUTPUT_DIR / "catboost_retrieval.cbm"))

display(
    pd.DataFrame({
        "feature": FEATURE_NAMES,
        "importance": cb_model.feature_importances_,
    }).sort_values("importance", ascending=False)
)

Обучающих пар: 584316
Положительных пар: 2880
Пропущено запросов без найденных положительных: 399
0:	learn: 0.5324873	total: 189ms	remaining: 1m 34s
100:	learn: 0.0242230	total: 1.12s	remaining: 4.42s
200:	learn: 0.0236549	total: 1.92s	remaining: 2.85s
300:	learn: 0.0232545	total: 2.7s	remaining: 1.78s
400:	learn: 0.0228613	total: 3.47s	remaining: 858ms
499:	learn: 0.0225173	total: 4.26s	remaining: 0us


,feature,importance
3,same_location,21.797443
7,log_reviews,12.836682
1,char_similarity,12.568989
0,word_similarity,11.993858
2,filter_similarity,11.000389
4,text_score,9.088160
8,log_price,8.008487
6,rating,6.704173
5,text_score_location4,3.669123
11,messages_forbidden,1.316020


### 18. Проверяем модель и смеси

Обучили CatBoost на 584 316 парах, из них 2 880 положительных. При подготовке обучения пропустили 399 запросов без найденных правильных кандидатов. Самым важным признаком оказалось совпадение локации.

Теперь сравним текстовый поиск, CatBoost и их смеси. В смеси учитываем место объявления в каждом списке: вклад равен 1 / (60 + место), затем складываем вклады с выбранными весами. Вес CatBoost 0 означает только текстовый поиск, 1 означает только модель. Также пробуем множители локации 4, 8 и 16.

Настройки выбираем по той же валидации, поэтому лучший результат здесь не считаем независимой оценкой качества на новых данных.

In [ ]:
def get_cb_scores(packs):
    lengths = [len(pack["indices"]) for pack in packs]
    features = np.vstack([pack["features"] for pack in packs])

    scores = cb_model.predict_proba(features)[:, 1]

    return np.split(scores, np.cumsum(lengths)[:-1])


valid_cb_scores = get_cb_scores(valid_packs)


def reciprocal_ranks(scores):
    order = np.argsort(-scores, kind="stable")
    ranks = np.empty(len(scores), dtype=np.float32)
    ranks[order] = np.arange(1, len(scores) + 1)

    return 1.0 / (60.0 + ranks)


def predict_mixture(packs, model_scores, location_multiplier, cb_weight):
    predictions = []

    for pack, cb_scores in zip(packs, model_scores):
        features = pack["features"]

        text_scores = features[:, 4]
        same_location = features[:, 3]

        lexical_scores = text_scores * (
            1 + (location_multiplier - 1) * same_location
        )

        if cb_weight == 0:
            scores = lexical_scores
        elif cb_weight == 1:
            scores = cb_scores
        else:
            scores = (
                (1 - cb_weight) * reciprocal_ranks(lexical_scores)
                + cb_weight * reciprocal_ranks(cb_scores)
            )

        chosen = top_indices(scores, 50)
        item_positions = pack["indices"][chosen]

        predictions.append(retrieval_ids[item_positions].tolist())

    return predictions


experiment_rows = []
experiment_predictions = {}

for multiplier in [4.0, 8.0, 16.0]:
    for cb_weight in [0.0, 0.25, 0.5, 0.75, 1.0]:
        # Мы проверяем чистый CatBoost один раз, потому что при весе CatBoost = 1 мы используем только оценки обученной модели;
        # множитель текстовой оценки на результат уже не влияет,
        if cb_weight == 1.0 and multiplier != 4.0:
            continue

        key = (multiplier, cb_weight)

        predictions = predict_mixture(
            valid_packs,
            valid_cb_scores,
            location_multiplier=multiplier,
            cb_weight=cb_weight,
        )

        experiment_predictions[key] = predictions

        experiment_rows.append({
            "location_multiplier": multiplier,
            "catboost_weight": cb_weight,
            "Recall@50": recall_at_50(predictions, valid_truth),
        })

experiments = (
    pd.DataFrame(experiment_rows)
    .sort_values("Recall@50", ascending=False)
    .reset_index(drop=True)
)

display(experiments)
print("Исходный baseline:", recall_at_50(
    valid_predictions[best_multiplier], valid_truth
))

,location_multiplier,catboost_weight,Recall@50
0,16.0,0.50,0.782933
1,8.0,0.50,0.781933
2,16.0,0.25,0.775433
3,16.0,0.00,0.774100
4,8.0,0.00,0.773600
5,16.0,0.75,0.771400
6,8.0,0.25,0.770433
7,8.0,0.75,0.769400
8,4.0,0.50,0.768433
9,4.0,0.75,0.765400


Исходный baseline: 0.7401


### 19. Выбираем итоговый вариант и сохраняем ответ

Лучший результат на валидации дала смесь с весом CatBoost 0,5 и множителем локации 16: Recall@50 = 0,7829. Для сравнения, чистый CatBoost дал 0,7477, текстовый поиск с множителем 16 дал 0,7741, а исходный baseline дал 0,7401. Вес 0,5 относится к сумме оценок по местам, а не к выбору 25 объявлений из каждого списка.

Применяем выбранный вариант к benchmark, проверяем формат и сохраняем answer.csv. Объявления, добавленные только для валидации, в финальный поиск не включаем. Таблицу экспериментов тоже сохраняем, чтобы остались результаты сравнения.

In [ ]:
best_row = experiments.iloc[0]

original_recall = recall_at_50(
    valid_predictions[best_multiplier],
    valid_truth,
)

if best_row["Recall@50"] > original_recall:
    print("Готовим ответы улучшенного варианта.")

    benchmark_packs = build_candidate_packs(
        queries,
        benchmark_only=True,
    )

    benchmark_cb_scores = get_cb_scores(benchmark_packs)

    final_predictions = predict_mixture(
        benchmark_packs,
        benchmark_cb_scores,
        location_multiplier=float(best_row["location_multiplier"]),
        cb_weight=float(best_row["catboost_weight"]),
    )

    print("Выбранные настройки:")
    print(best_row)

else:
    print("Прироста нет — сохраняем исходный baseline.")
    final_predictions = benchmark_predictions

final_answer = pd.DataFrame({
    "query_id": queries["query_id"],
    "answer": [" ".join(ids) for ids in final_predictions],
})

allowed_ids = set(items["item_id"])

assert len(final_answer) == len(queries)
assert final_answer["query_id"].is_unique
assert set(final_answer["query_id"]) == set(queries["query_id"])

for ids in final_predictions:
    assert len(ids) == 50
    assert len(ids) == len(set(ids))
    assert set(ids) <= allowed_ids

final_answer.to_csv(
    OUTPUT_DIR / "answer.csv",
    index=False,
    encoding="utf-8",
)

experiments.to_csv(
    OUTPUT_DIR / "validation_experiments.csv",
    index=False,
)

print("Готово:", OUTPUT_DIR / "answer.csv")

Готовим ответы улучшенного варианта.


  0%|          | 0/77 [00:00<?, ?it/s]

Выбранные настройки:
location_multiplier    16.000000
catboost_weight         0.500000
Recall@50               0.782933
Name: 0, dtype: float64
Готово: /kaggle/working/answer.csv


### Итог

На Stepik baseline получил 0,690, а итоговый вариант 0,738. Прирост составил 4,8 процентного пункта Recall@50.

Файл для отправки: answer.csv. Дополнительно сохраняются answer_baseline.csv, модель catboost_retrieval.cbm и таблица validation_experiments.csv.